# DentoSeg-SSL — Unified Run Notebook

This notebook runs the full DentoSeg-SSL pipeline end-to-end.
It auto-detects the compute environment (Kaggle / Google Colab / Local)
and adjusts paths and install steps accordingly.

**Stages**
1. Environment setup & dependency install
2. Repository clone / path wiring
3. Configuration
4. Contrastive SSL pretraining
5. Supervised segmentation fine-tuning
6. Evaluation & visualisation

## 1 · Environment Detection & Dependency Install

In [ ]:
import os, sys, subprocess, pathlib

# ── Detect environment ─────────────────────────────────────────────
def detect_env():
    if os.path.exists('/kaggle'):
        return 'kaggle'
    if 'COLAB_GPU' in os.environ or 'COLAB_BACKEND_URL' in os.environ:
        return 'colab'
    return 'local'

ENV = detect_env()
print(f'Detected environment: {ENV}')

# ── Install dependencies ────────────────────────────────────────────
# On Kaggle/Colab, TF is pre-installed; only missing packages need adding.
EXTRA_PACKAGES = ['natsort', 'pyyaml', 'tqdm', 'opencv-python-headless']

if ENV in ('kaggle', 'colab'):
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + EXTRA_PACKAGES,
        check=True
    )
    print('Extra packages installed.')
else:
    print('Local env — ensure you have run: pip install -r requirements.txt')

## 2 · Clone Repository / Wire Project Root

In [ ]:
REPO_URL  = 'https://github.com/YOUR_USERNAME/DentoSeg-SSL-Self-Supervised-Dental-Panoramic-Segmentation.git'
REPO_NAME = 'DentoSeg-SSL-Self-Supervised-Dental-Panoramic-Segmentation'

if ENV == 'kaggle':
    # On Kaggle, clone into /kaggle/working
    REPO_ROOT = pathlib.Path('/kaggle/working') / REPO_NAME
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_ROOT)], check=True)
        print(f'Repo cloned → {REPO_ROOT}')
    else:
        print(f'Repo already present at {REPO_ROOT}')

elif ENV == 'colab':
    # On Colab, clone into /content
    REPO_ROOT = pathlib.Path('/content') / REPO_NAME
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_ROOT)], check=True)
        print(f'Repo cloned → {REPO_ROOT}')
    else:
        print(f'Repo already present at {REPO_ROOT}')

else:
    # Local: the notebook is already inside the repo
    REPO_ROOT = pathlib.Path('..').resolve()
    print(f'Local run — project root: {REPO_ROOT}')

# Add project root to Python path so all modules are importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('sys.path updated.')

## 3 · Imports & Seed

In [ ]:
import tensorflow as tf
import numpy as np

from data.dataset       import DentalDataset
from data.augmentation  import RandomContrast
from models.encoder     import create_encoder
from models.segmentation import create_contrastive_model, create_segmentation_model
from training.pretrain  import ContrastiveTrainer
from training.finetune  import train_segmentation
from evaluation.metrics import iou_metric, dice_coefficient
from evaluation.visualize import visualize_results, plot_training_history
from utils.env          import detect_environment, get_default_paths
from utils.seed         import set_global_seed

print('TensorFlow version:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 4 · Configuration

Edit the values in this cell to change any hyperparameter.
Paths are auto-populated from the environment; override them if needed.

In [ ]:
SEED = 42
set_global_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────
paths = get_default_paths(ENV)

IMAGES_PATH = paths['images']   # override here if needed
MASKS_PATH  = paths['masks']
OUTPUT_DIR  = paths['output']

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Images : {IMAGES_PATH}')
print(f'Masks  : {MASKS_PATH}')
print(f'Outputs: {OUTPUT_DIR}')

# ── Hyperparameters ────────────────────────────────────────────────
IMAGE_SIZE   = 224
CHANNELS     = 1
INPUT_SHAPE  = (IMAGE_SIZE, IMAGE_SIZE, CHANNELS)

# Pretraining
BATCH_SIZE_PRETRAIN   = 16
EPOCHS_PRETRAIN       = 50
LR_PRETRAIN           = 3e-4
TEMPERATURE           = 0.1
PROJECTION_DIM        = 128

# Fine-tuning
BATCH_SIZE_FINETUNE   = 16
EPOCHS_FINETUNE       = 50
LR_FINETUNE           = 1e-4
FREEZE_ENCODER_EPOCHS = 10

# Augmentation
AUG_CFG = dict(
    horizontal_flip      = True,
    rotation_factor      = 0.1,
    gaussian_noise_stddev= 0.05,
    contrast_lower       = 0.85,
    contrast_upper       = 1.15,
)

# Output paths
ENCODER_CKPT   = f'{OUTPUT_DIR}/pretrain/encoder_pretrained.keras'
BEST_MODEL     = f'{OUTPUT_DIR}/finetune/best_model.keras'
FINAL_MODEL    = f'{OUTPUT_DIR}/finetune/dental_segmentation_model.keras'
RESULTS_PLOT   = f'{OUTPUT_DIR}/finetune/segmentation_results.png'
HISTORY_PLOT   = f'{OUTPUT_DIR}/finetune/training_history.png'

print('Configuration ready.')

## 5 · Load Data

In [ ]:
dental_dataset = DentalDataset(IMAGES_PATH, MASKS_PATH, image_size=IMAGE_SIZE)
(train_images, train_masks), (test_images, test_masks) = dental_dataset.prepare_data(
    test_ratio=0.2,
    seed=SEED,
)

print(f'Train: {train_images.shape}  Test: {test_images.shape}')

## 6 · Contrastive SSL Pretraining

In [ ]:
# Build encoder and contrastive model
encoder           = create_encoder(input_shape=INPUT_SHAPE)
contrastive_model = create_contrastive_model(
    encoder        = encoder,
    input_shape    = INPUT_SHAPE,
    projection_dim = PROJECTION_DIM,
    augmentation_cfg = AUG_CFG,
)
contrastive_model.summary()

In [ ]:
# Build tf.data pipeline (images only — SSL needs no labels)
pretrain_ds = (
    tf.data.Dataset.from_tensor_slices(train_images)
    .shuffle(len(train_images), seed=SEED)
    .batch(BATCH_SIZE_PRETRAIN)
    .prefetch(tf.data.AUTOTUNE)
)

trainer = ContrastiveTrainer(
    model         = contrastive_model,
    encoder       = encoder,
    temperature   = TEMPERATURE,
    learning_rate = LR_PRETRAIN,
    log_path      = f'{OUTPUT_DIR}/pretrain/pretrain_log.csv',
)

trainer.train(
    dataset            = pretrain_ds,
    epochs             = EPOCHS_PRETRAIN,
    encoder_checkpoint = ENCODER_CKPT,
)

## 7 · Segmentation Fine-tuning

If you already have a pretrained encoder checkpoint, skip Section 6 and run
the cell below to load it directly.

In [ ]:
import pathlib as _pl

if _pl.Path(ENCODER_CKPT).exists():
    print(f'Loading pretrained encoder from {ENCODER_CKPT} …')
    encoder = tf.keras.models.load_model(
        ENCODER_CKPT,
        custom_objects={'RandomContrast': RandomContrast},
    )
else:
    print('No checkpoint found — using current encoder (from pretraining above).')

seg_model = create_segmentation_model(encoder=encoder, input_shape=INPUT_SHAPE)
seg_model.summary()

In [ ]:
history = train_segmentation(
    model                  = seg_model,
    encoder                = encoder,
    train_images           = train_images,
    train_masks            = train_masks,
    val_ratio              = 0.2,
    batch_size             = BATCH_SIZE_FINETUNE,
    epochs                 = EPOCHS_FINETUNE,
    learning_rate          = LR_FINETUNE,
    freeze_encoder_epochs  = FREEZE_ENCODER_EPOCHS,
    best_model_path        = BEST_MODEL,
    final_model_path       = FINAL_MODEL,
    log_csv_path           = f'{OUTPUT_DIR}/finetune/finetune_log.csv',
    seed                   = SEED,
)

## 8 · Evaluation

In [ ]:
from training.losses import dice_loss

# Re-compile with named metrics so evaluate() returns labelled results
seg_model.compile(
    optimizer = tf.keras.optimizers.Adam(),
    loss      = dice_loss,
    metrics   = [
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        iou_metric,
        dice_coefficient,
    ],
)

results = seg_model.evaluate(test_images, test_masks, verbose=1)
metric_names = ['Loss', 'Accuracy', 'Precision', 'Recall', 'IoU', 'Dice']
print('\n── Test Results ──────────────────────────')
for name, val in zip(metric_names, results):
    print(f'  {name:<12}: {val:.4f}')

## 9 · Visualisation

In [ ]:
plot_training_history(history, save_path=HISTORY_PLOT)

In [ ]:
visualize_results(
    seg_model, test_images, test_masks,
    num_samples = 5,
    save_path   = RESULTS_PLOT,
    seed        = SEED,
)

## 10 · Load a Saved Model (Inference Only)

Use this cell to skip training entirely and load a previously saved model.

In [ ]:
from training.losses import dice_loss  # noqa: F811

MODEL_TO_LOAD = FINAL_MODEL  # or BEST_MODEL

loaded_model = tf.keras.models.load_model(
    MODEL_TO_LOAD,
    custom_objects={
        'RandomContrast' : RandomContrast,
        'dice_loss'      : dice_loss,
        'iou_metric'     : iou_metric,
        'dice_coefficient': dice_coefficient,
    },
)
print(f'Model loaded from {MODEL_TO_LOAD}')
loaded_model.summary()